<a href="https://colab.research.google.com/github/eaedk/Machine-Learning-Tutorials-Updated/blob/main/03_computer_vision/06_semantic_segmentation_quest_shapes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎨 Quest · Semantic segmentation: colour every pixel

Detection draws boxes. **Semantic segmentation** goes further: it gives a class to **every single pixel** of the
image. Road or sidewalk? Tumour or healthy tissue? Forest, field or city on a satellite image?

**A short history**
- **2015 · U-Net.** Olaf Ronneberger and colleagues (University of Freiburg) design a network to segment cells in
  microscope images, with very few training images. They win the ISBI cell tracking challenge by a large margin. Ten
  years later, U-Net is still everywhere: medical imaging, satellite images… and inside image generators like Stable
  Diffusion!
- **2016 · Cityscapes.** 5 000 finely annotated street images from 50 cities (annotating one image took ~1.5 hours!)
  become the benchmark for self-driving cars.
- **Today.** Transformers (SegFormer, Mask2Former), foundation models like SAM, and fast YOLO26 semantic models.

**Your mission:** build a U-Net **from scratch** on a toy dataset that you generate yourself (no download!), learn
why **IoU** beats pixel accuracy, then try a pre-trained YOLO26 street-scene model.

| You will practise | Concept |
|---|---|
| Generating images + masks | what a segmentation label is |
| Encoder / decoder / skip connections | the U-Net architecture |
| Per-pixel classification loss | cross-entropy on every pixel |
| Pixel accuracy vs IoU vs Dice | choosing the right metric |

In [ ]:
# ⚙️ Setup: run me first! Finds the course toolkit `mlquest` (Windows, macOS, Linux and Colab).
import os, sys, subprocess, pathlib
if "google.colab" in sys.modules:  # on Colab: grab the repo (once) and the few missing packages
    if not pathlib.Path("/content/Machine-Learning-Tutorials-Updated").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/eaedk/Machine-Learning-Tutorials-Updated", "/content/Machine-Learning-Tutorials-Updated"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)
    os.chdir("/content/Machine-Learning-Tutorials-Updated")
ROOT = pathlib.Path.cwd()
while not (ROOT / "mlquest").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from mlquest import *  # paths, seed_everything, get_device, quest/checkpoint, Leaderboard...
set_style()
SEED = seed_everything(42)

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import cv2
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

DEVICE = get_device()
quest("Colour every pixel", chapter="Track 3 · Computer vision · Semantic segmentation", xp=300)

## 1. Generate your own dataset 🖌️

Each sample is a 96×96 noisy image with a few random shapes, and its **mask**: an image of the same size where every
pixel holds a class id:

| id | class |
|---|---|
| 0 | background |
| 1 | circle |
| 2 | square |
| 3 | triangle |

### 🎯 Exercise 1: complete the generator by drawing the squares
We use OpenCV drawing functions. Draw the square **in the image** (with `color`) **and in the mask** (with the class id
`2`). Hint: `cv2.rectangle(array, (x - r, y - r), (x + r, y + r), value, thickness=-1)` (`-1` = filled).

In [ ]:
CLASSES = ["background", "circle", "square", "triangle"]
SIZE = 96

def make_sample(rng):
    img = rng.normal(0.5, 0.12, (SIZE, SIZE, 3)).astype(np.float32)   # noisy background
    mask = np.zeros((SIZE, SIZE), np.uint8)
    for _ in range(rng.integers(1, 4)):                                 # 1 to 3 shapes
        shape = rng.integers(1, 4)
        x, y, r = rng.integers(16, SIZE - 16), rng.integers(16, SIZE - 16), int(rng.integers(7, 16))
        color = tuple(float(c) for c in rng.uniform(0, 1, 3))
        if shape == 1:
            cv2.circle(img, (x, y), r, color, -1); cv2.circle(mask, (x, y), r, 1, -1)
        elif shape == 2:
            ...   # TODO: draw a filled square in img (color) and in mask (value 2)
        else:
            pts = np.array([[x, y - r], [x - r, y + r], [x + r, y + r]], np.int32)
            cv2.fillPoly(img, [pts], color); cv2.fillPoly(mask, [pts], 3)
    return np.clip(img, 0, 1), mask

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
CLASSES = ["background", "circle", "square", "triangle"]
SIZE = 96

def make_sample(rng):
    img = rng.normal(0.5, 0.12, (SIZE, SIZE, 3)).astype(np.float32)
    mask = np.zeros((SIZE, SIZE), np.uint8)
    for _ in range(rng.integers(1, 4)):
        shape = rng.integers(1, 4)
        x, y, r = rng.integers(16, SIZE - 16), rng.integers(16, SIZE - 16), int(rng.integers(7, 16))
        color = tuple(float(c) for c in rng.uniform(0, 1, 3))
        if shape == 1:
            cv2.circle(img, (x, y), r, color, -1); cv2.circle(mask, (x, y), r, 1, -1)
        elif shape == 2:
            cv2.rectangle(img, (x - r, y - r), (x + r, y + r), color, -1)
            cv2.rectangle(mask, (x - r, y - r), (x + r, y + r), 2, -1)
        else:
            pts = np.array([[x, y - r], [x - r, y + r], [x + r, y + r]], np.int32)
            cv2.fillPoly(img, [pts], color); cv2.fillPoly(mask, [pts], 3)
    return np.clip(img, 0, 1), mask
```

</details>

In [ ]:
rng = np.random.default_rng(SEED)
samples = [make_sample(rng) for _ in range(200)]
all_ids = set(np.unique(np.concatenate([m.ravel() for _, m in samples])))
check(all_ids == {0, 1, 2, 3}, "All 4 classes appear in the masks. Your dataset factory works!",
      f"Classes found in masks: {sorted(all_ids)}. Did you draw value 2 in the mask?")

CMAP = ListedColormap(["#1f2937", "#f97316", "#2563eb", "#16a34a"])
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for k in range(6):
    axes[0, k].imshow(samples[k][0]); axes[1, k].imshow(samples[k][1], cmap=CMAP, vmin=0, vmax=3)
    axes[0, k].axis("off"); axes[1, k].axis("off")
axes[0, 0].set_title("images", loc="left"); axes[1, 0].set_title("masks (one colour per class)", loc="left")
plt.tight_layout(); plt.show()

Notice: the shapes have **random colours**, so colour alone can't tell a circle from a square. The network must learn
**shapes**: that's what convolutions are good at.

In [ ]:
class ShapesDataset(Dataset):
    """Generates samples on the fly: an infinite supply of training data!"""
    def __init__(self, n, seed):
        rng = np.random.default_rng(seed)
        self.data = [make_sample(rng) for _ in range(n)]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, i):
        img, mask = self.data[i]
        return torch.from_numpy(img).permute(2, 0, 1), torch.from_numpy(mask).long()

train_dl = DataLoader(ShapesDataset(3000, seed=1), batch_size=32, shuffle=True, num_workers=NUM_WORKERS)
val_dl = DataLoader(ShapesDataset(500, seed=2), batch_size=64, num_workers=NUM_WORKERS)

## 2. The U-Net 🏗️

```
 input 3×96×96                                                      output 4×96×96 (a score per class per pixel)
   │ enc1 (32ch, 96×96) ───────────── skip ─────────────▶ concat ─▶ dec1 (32ch, 96×96)
   ▼ pool                                                   ▲ up
   │ enc2 (64ch, 48×48) ─────── skip ───────▶ concat ─▶ dec2 (64ch, 48×48)
   ▼ pool                                         ▲ up
   └──────────▶ bottleneck (128ch, 24×24) ────────┘
```

- The **encoder** (left) shrinks the image and grows the number of channels: it understands *what* is there.
- The **decoder** (right) upsamples back to full resolution: it decides *where*, pixel by pixel.
- The **skip connections** copy fine details from the encoder to the decoder (concatenation). Without them, edges
  come out blurry. They are U-Net's key idea.

### 🎯 Exercise 2: wire the skip connections in `forward`
`torch.cat([a, b], dim=1)` concatenates two feature maps along the channel dimension.

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
    )

class TinyUNet(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.enc1, self.enc2 = conv_block(3, 32), conv_block(32, 64)
        self.bottleneck = conv_block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = conv_block(128, 64)          # 64 (upsampled) + 64 (skip from enc2)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = conv_block(64, 32)           # 32 (upsampled) + 32 (skip from enc1)
        self.head = nn.Conv2d(32, n_classes, 1)  # 1×1 conv: 32 features → 4 class scores per pixel

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.dec2(...)   # TODO: concatenate self.up2(b) with the skip e2
        d1 = self.dec1(...)   # TODO: concatenate self.up1(d2) with the skip e1
        return self.head(d1)

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
    )

class TinyUNet(nn.Module):
    def __init__(self, n_classes=4):
        super().__init__()
        self.enc1, self.enc2 = conv_block(3, 32), conv_block(32, 64)
        self.bottleneck = conv_block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = conv_block(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = conv_block(64, 32)
        self.head = nn.Conv2d(32, n_classes, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)
```

</details>

In [ ]:
net = TinyUNet()
x, y = next(iter(val_dl))
out = net(x[:2])
check(tuple(out.shape) == (2, 4, 96, 96),
      f"Output (2, 4, 96, 96): 4 scores for each of the 96×96 pixels. {sum(p.numel() for p in net.parameters()):,} parameters.",
      "The output must have the same height/width as the input, with 4 channels.")

## 3. Metrics: why pixel accuracy lies 🤥

**Pixel accuracy** = share of pixels with the right class. Sounds natural… but most pixels are background!

**IoU per class** (also called the *Jaccard index*) = |prediction ∩ truth| / |prediction ∪ truth|, computed on the
pixels of that class. **mIoU** = mean over classes. **Dice** = 2·|P ∩ T| / (|P| + |T|), a close cousin, very popular in
medical imaging.

### 🎯 Exercise 3: implement IoU and Dice for one class

In [ ]:
def iou_dice(pred, true, cls):
    """pred, true: integer tensors of class ids, same shape. Returns (IoU, Dice) for class `cls`."""
    p, t = pred == cls, true == cls
    inter = (p & t).sum().item()
    union = ...                       # TODO: (p | t).sum().item()
    total = p.sum().item() + t.sum().item()
    iou = inter / union if union else float("nan")
    dice = ...                        # TODO: 2 * inter / total (nan if total == 0)
    return iou, dice

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
def iou_dice(pred, true, cls):
    p, t = pred == cls, true == cls
    inter = (p & t).sum().item()
    union = (p | t).sum().item()
    total = p.sum().item() + t.sum().item()
    iou = inter / union if union else float("nan")
    dice = 2 * inter / total if total else float("nan")
    return iou, dice
```

</details>

In [ ]:
a = torch.tensor([[1, 1, 0, 0]]); b = torch.tensor([[1, 0, 0, 0]])
check(np.allclose(iou_dice(a, b, 1), (0.5, 2 / 3)), "IoU = 1/2 and Dice = 2/3 on the toy example. Correct!",
      "On pred=[1,1,0,0], true=[1,0,0,0], class 1: IoU should be 0.5 and Dice 0.667.")

def evaluate(model_fn, dl):
    preds, trues = [], []
    for x, y in dl:
        preds.append(model_fn(x)); trues.append(y)
    p, t = torch.cat(preds), torch.cat(trues)
    res = {"pixel_acc": (p == t).float().mean().item()}
    ious = [iou_dice(p, t, c)[0] for c in range(4)]
    res.update({f"IoU_{CLASSES[c]}": ious[c] for c in range(4)})
    res["mIoU"] = float(np.nanmean(ious))
    return res

lazy = evaluate(lambda x: torch.zeros(x.shape[0], SIZE, SIZE, dtype=torch.long), val_dl)   # always "background"
lb = Leaderboard("shapes_segmentation", metric="mIoU")
lb.add("Lazy model (all background)", lazy["mIoU"], pixel_acc=round(lazy["pixel_acc"], 3))
pd.Series(lazy).round(3)

The lazy model has a great pixel accuracy (most pixels *are* background) while it found **zero** shapes. IoU reveals
the truth. 👉 In segmentation competitions, the metric is almost always mIoU or Dice, never plain accuracy.

In [ ]:
checkpoint("semseg/metrics", xp=60)

## 4. Train the U-Net 🏋️

The loss is a **cross-entropy computed on every pixel**, then averaged: segmentation is just classification, 9 216
times per image.

In [ ]:
def train_unet(model, epochs=6, lr=2e-3):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    model.curve = []                                  # mIoU after each epoch, to compare learning speed
    for epoch in range(1, epochs + 1):
        model.train(); t0 = time.time(); total = 0
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step()
            total += loss.item() * len(x)
        model.eval()
        with torch.no_grad():
            res = evaluate(lambda x: model(x.to(DEVICE)).argmax(1).cpu(), val_dl)
        model.curve.append(res["mIoU"])
        print(f"epoch {epoch} · loss {total / len(train_dl.dataset):.3f} · pixel acc {res['pixel_acc']:.3f} · mIoU {res['mIoU']:.3f} · {time.time() - t0:.0f}s")
    return res

seed_everything(SEED)
unet = TinyUNet()
res = train_unet(unet)
lb.add("TinyUNet (6 epochs)", res["mIoU"], pixel_acc=round(res["pixel_acc"], 3))
lb.show()

In [ ]:
x, y = next(iter(val_dl))
with torch.no_grad():
    pred = unet.eval()(x.to(DEVICE)).argmax(1).cpu()
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for k in range(6):
    axes[0, k].imshow(x[k].permute(1, 2, 0)); axes[1, k].imshow(y[k], cmap=CMAP, vmin=0, vmax=3)
    axes[2, k].imshow(pred[k], cmap=CMAP, vmin=0, vmax=3)
for ax in axes.ravel():
    ax.axis("off")
for r, t in enumerate(["image", "truth", "prediction"]):
    axes[r, 0].set_title(t, loc="left")
plt.tight_layout(); plt.show()
check(res["mIoU"] > 0.7, "Your U-Net separates circles, squares and triangles. Shapes, not colours!",
      "mIoU should exceed 0.7 after 6 epochs. Check the skip connections.")

### 🎯 Exercise 4: what do skip connections bring?
Create `NoSkipUNet`, a copy of `TinyUNet` **without** skip connections (the decoders only receive the upsampled
features), train it with the same settings, and add it to the leaderboard. What happens to the edges?

In [ ]:
class NoSkipUNet(TinyUNet):
    def __init__(self, n_classes=4):
        super().__init__(n_classes)
        self.dec2 = conv_block(64, 64)    # no skip → only 64 input channels
        self.dec1 = conv_block(32, 32)

    def forward(self, x):
        b = self.bottleneck(self.pool(self.enc2(self.pool(self.enc1(x)))))
        d2 = ...   # TODO: self.dec2 applied to self.up2(b) (no concatenation)
        d1 = ...   # TODO
        return self.head(d1)

seed_everything(SEED)
noskip = NoSkipUNet()
res_noskip = train_unet(noskip)
lb.add("U-Net without skips", res_noskip["mIoU"], pixel_acc=round(res_noskip["pixel_acc"], 3))
lb.show()

<details>
<summary>💡 <b>Stuck? Click to reveal one possible solution</b></summary>

```python
class NoSkipUNet(TinyUNet):
    def __init__(self, n_classes=4):
        super().__init__(n_classes)
        self.dec2 = conv_block(64, 64)
        self.dec1 = conv_block(32, 32)

    def forward(self, x):
        b = self.bottleneck(self.pool(self.enc2(self.pool(self.enc1(x)))))
        d2 = self.dec2(self.up2(b))
        d1 = self.dec1(self.up1(d2))
        return self.head(d1)

seed_everything(SEED)
noskip = NoSkipUNet()
res_noskip = train_unet(noskip)
lb.add("U-Net without skips", res_noskip["mIoU"], pixel_acc=round(res_noskip["pixel_acc"], 3))
lb.show()
```

</details>

In [ ]:
pd.DataFrame({"with skips": unet.curve, "without skips": noskip.curve}, index=range(1, 7)).plot(
    marker="o", xlabel="epoch", ylabel="validation mIoU", title="Skip connections speed up learning")
plt.show()

Without skips, the decoder must rebuild edges from a 24×24 summary: it learns noticeably **slower** (compare the first
epochs) and its borders stay a bit blurrier. On our simple shapes it eventually catches up; on real images with fine
structures (vessels, cracks, thin poles), the gap stays large. That's why every modern segmentation network has some
form of skip connection.

## 5. Real streets: a pre-trained YOLO26 semantic model 🚗

Toy shapes are great to learn; let's finish with the real thing. YOLO26n-sem was trained on **Cityscapes** (19
classes: road, sidewalk, car, person, vegetation, sky…). We run it on a few Cityscapes validation images (17 MB
download).

In [ ]:
import shutil
from pathlib import Path
os.environ["YOLO_VERBOSE"] = "False"
from ultralytics import YOLO
from ultralytics.utils import YAML
from ultralytics.utils.checks import check_yaml

cfg = YAML.load(check_yaml("cityscapes8.yaml"))
city = DATA_DIR / "yolo_datasets" / "cityscapes8"
if not city.exists():
    archive = download(cfg["download"], DATA_DIR / "yolo_datasets" / "cityscapes8.zip")
    extract(archive, DATA_DIR / "yolo_datasets"); archive.unlink()

sem_model = YOLO(str(DATA_DIR / "weights" / "yolo26n-sem.pt"))
img_paths = sorted((city / "images" / "val").glob("*.png"))[:3]
results = sem_model.predict(img_paths, imgsz=1024, device=DEVICE)

# Ground-truth masks store raw Cityscapes ids: map them to the 19 training classes (255 = ignored)
lut = np.full(256, 255, np.uint8)
for raw, train_id in cfg["label_mapping"].items():
    if isinstance(train_id, int) and raw >= 0:
        lut[raw] = train_id

city_cmap = plt.get_cmap("tab20", 20)
fig, axes = plt.subplots(len(img_paths), 3, figsize=(15, 3 * len(img_paths)))
scores = []
for k, (path, r) in enumerate(zip(img_paths, results)):
    pred = r.semantic_mask.data.cpu().numpy()
    gt = lut[cv2.imread(str(city / "masks" / "val" / path.name), cv2.IMREAD_GRAYSCALE)]
    valid = gt != 255
    ious = [iou_dice(torch.from_numpy(pred[valid]), torch.from_numpy(gt[valid]), c)[0] for c in np.unique(gt[valid])]
    scores.append(np.nanmean(ious))
    axes[k, 0].imshow(cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB))
    axes[k, 1].imshow(np.where(valid, gt, 19), cmap=city_cmap, vmin=0, vmax=19)
    axes[k, 2].imshow(pred, cmap=city_cmap, vmin=0, vmax=19)
    axes[k, 2].set_title(f"prediction · mIoU {scores[-1]:.2f}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
axes[0, 0].set_title("image"); axes[0, 1].set_title("truth (19 classes)")
plt.tight_layout(); plt.show()
names = sem_model.names
present = np.unique(results[0].semantic_mask.data.cpu().numpy())
print("Classes found in the first image:", ", ".join(names[int(c)] for c in present))

## 🏁 Quest complete!

You generated a dataset, built a U-Net with skip connections, learnt why IoU and Dice matter and used a real street
segmentation model.

### 🚀 Your turn (bonus XP)
1. Add a 5th class (ellipses with `cv2.ellipse`) and retrain.
2. Make the task harder: overlapping shapes, more noise, smaller shapes. When does the U-Net start failing?
3. Replace cross-entropy by a **Dice loss** (1 − soft Dice computed from the softmax probabilities).
4. Add a third level to the U-Net (enc3 / dec3). Does it help on 96×96 images?

Next: `07_semantic_segmentation_app_polyps.ipynb`, a medical application.

In [ ]:
checkpoint("semseg/complete", xp=240)
badge("Pixel Painter", "🎨")
progress()